# 🏠 House Price Prediction with MLflow
## Real-World Machine Learning with Hyperparameter Optimization

This notebook demonstrates **advanced MLflow features** using a real-world problem:
**Predict California housing prices based on location, bedrooms, and other features**

### What You'll Learn:
1. Load a real dataset (California Housing)
2. Perform **hyperparameter tuning** - automatically finding the best model settings
3. Compare model results with **GridSearchCV** (tests 24 different combinations!)
4. Track every experiment in MLflow for easy comparison
5. Register the best model for future use

### Why This Matters:
In real ML projects, finding the right hyperparameters can mean the difference between 70% and 95% accuracy!


## 📚 Tutorial Overview

In this notebook, we will:

1. **Load Real Data** - California Housing Dataset (20,000+ houses)
2. **Run Hyperparameter Tuning** - Test 24 different model configurations automatically
3. **Track Everything in MLflow** - Log all parameters and metrics to a database
4. **Compare Results** - See which hyperparameters produced the best accuracy
5. **Register Best Model** - Save the winning model for future use

**Key Skill:** GridSearchCV automatically tests combinations of hyperparameters so we don't have to manually try each one!


In [1]:
# ============================================================================
# STEP 1: Import Libraries
# ============================================================================

import pandas as pd                              # Data manipulation
import mlflow                                  # Experiment tracking
import mlflow.sklearn                          # MLflow integration with scikit-learn
from sklearn.ensemble import RandomForestRegressor  # The ML algorithm we'll use
from sklearn.model_selection import train_test_split, GridSearchCV  # Data splitting & hyperparameter tuning
from sklearn.metrics import mean_squared_error  # Metric to evaluate model
from sklearn.datasets import fetch_california_housing  # Dataset

print("✅ All libraries imported successfully!")

# ============================================================================
# STEP 2: Load the California Housing Dataset
# ============================================================================
# This dataset contains information about 20,640 houses in California
# Features: MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude
# Target: Median house prices (in units of $100,000)

housing = fetch_california_housing()  # Download the dataset
print(f"✅ Dataset loaded!")
print(f"   - Number of samples: {housing.data.shape[0]}")
print(f"   - Number of features: {housing.data.shape[1]}")

✅ All libraries imported successfully!
✅ Dataset loaded!
   - Number of samples: 20640
   - Number of features: 8


In [2]:
# ============================================================================
# STEP 3: Prepare the Dataset
# ============================================================================
# Convert the raw data into a nice pandas DataFrame with column names

# Create DataFrame with feature names as columns
data = pd.DataFrame(housing.data, columns=housing.feature_names)
# Add the target (price) as a new column
data["Price"] = housing.target

print("📊 First 10 rows of data:")
print(data.head(10))
print(f"\n✅ Dataset prepared!")
print(f"   Shape: {data.shape[0]} rows × {data.shape[1]} columns")
print(f"   Price range: ${housing.target.min()*100000:.0f} to ${housing.target.max()*100000:.0f}")

📊 First 10 rows of data:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   
5  4.0368      52.0  4.761658   1.103627       413.0  2.139896     37.85   
6  3.6591      52.0  4.931907   0.951362      1094.0  2.128405     37.84   
7  3.1200      52.0  4.797527   1.061824      1157.0  1.788253     37.84   
8  2.0804      42.0  4.294118   1.117647      1206.0  2.026891     37.84   
9  3.6912      52.0  4.970588   0.990196      1551.0  2.172269     37.84   

   Longitude  Price  
0    -122.23  4.526  
1    -122.22  3.585  
2    -122.24  3.521  
3    -122.25  3.413  
4    -122.25  3.422  
5    -

### Train Test Split, Model Hyperparameter Tuning, MLFLOW Experiments

In this section, we will:
1. **Split the data** - Divide our data into training (80%) and testing (20%) sets
2. **Perform hyperparameter tuning** - Test different model settings to find the best ones
3. **Track experiments with MLFlow** - Log all our experiments so we can compare results later

In [3]:
# ============================================================================
# STEP 4: Separate Features (X) and Target (y)
# ============================================================================
# X = **Features** (independent variables): What we know about the house
#     - Median income, house age, number of rooms, location, etc.
# y = **Target** (dependent variable): What we want to predict
#     - House price (this is what the model will predict)

from urllib.parse import urlparse

# Create feature matrix X (all columns EXCEPT Price)
X = data.drop(columns=["Price"])
print(f"✅ Features (X) shape: {X.shape}")
print(f"   Features: {list(X.columns)}")

# Create target vector y (only Price column)
y = data["Price"]
print(f"\n✅ Target (y) shape: {y.shape}")
print(f"   Target (y) statistics:")
print(f"   - Mean: ${y.mean()*100000:.0f}")
print(f"   - Std: ${y.std()*100000:.0f}")

✅ Features (X) shape: (20640, 8)
   Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

✅ Target (y) shape: (20640,)
   Target (y) statistics:
   - Mean: $206856
   - Std: $115396


In [4]:
# ============================================================================
# STEP 5: Define Hyperparameter Tuning Function
# ============================================================================
# This function will test many hyperparameter combinations and find the best one

def hyperparameter_tuning(X_train, y_train, param_grid):
    """
    Use GridSearchCV to find the best hyperparameters for Random Forest
    
    GridSearchCV:
    - Tests ALL combinations of hyperparameters
    - Uses cross-validation to make sure results are reliable
    - Returns the best model found
    
    Parameters:
    -----------
    X_train: Training features
    y_train: Training target values
    param_grid: Dictionary of hyperparameters to test
    
    Returns:
    --------
    grid_search: GridSearchCV object with best model and parameters
    """
    
    # Create a Random Forest Regressor
    # Random Forest = ensemble of decision trees that vote on predictions
    rf = RandomForestRegressor()
    
    # Define GridSearchCV parameters
    # cv=3: Use 3-fold cross-validation (split data 3 ways, test on each)
    # n_jobs=-1: Use ALL CPU cores to speed up (run tests in parallel)
    # verbose=2: Print detailed progress updates
    # scoring="neg_mean_squared_error": Lower is better (negative MSE)
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=3,
        n_jobs=-1,
        verbose=2,
        scoring="neg_mean_squared_error"
    )
    
    print("🔍 Starting GridSearchCV...")
    print(f"   Will test {len(param_grid['n_estimators']) * len(param_grid['max_depth']) * len(param_grid['min_samples_split']) * len(param_grid['min_samples_leaf'])} combinations")
    
    grid_search.fit(X_train, y_train)  # Run the search (takes a minute or so)
    
    print(f"✅ GridSearchCV complete!")
    return grid_search

#### What is Hyperparameter Tuning?
Hyperparameters are settings we choose before training our model. Think of them like the recipe ingredients:
- **n_estimators**: How many trees to use in the forest (more trees = more accurate but slower)
- **max_depth**: How deep each tree can grow (deeper = memorizes more but might overfit)
- **min_samples_split**: Minimum samples needed to split a node
- **min_samples_leaf**: Minimum samples needed at a leaf node

GridSearchCV tries ALL combinations of these settings and finds which one works best!

In [5]:
# ============================================================================
# STEP 6: Split Data, Define Parameters, and Train with MLflow Tracking
# ============================================================================

print("📝 Starting the complete machine learning pipeline...\n")

## Step 6a: Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"✅ Data split:")
print(f"   Training samples: {len(X_train)}")
print(f"   Testing samples: {len(X_test)}\n")

## Step 6b: Create model signature (tells MLflow what data format to expect)
from mlflow.models import infer_signature
signature = infer_signature(X_train, y_train)

## Step 6c: Define the hyperparameters to test
# This creates 2 × 3 × 2 × 2 = 24 total combinations!
param_grid = {
    "n_estimators": [100, 200],        # Number of trees in the forest
    "max_depth": [5, 10, None],        # How deep trees can grow (None = unlimited)
    "min_samples_split": [2, 5],       # Min samples to split a node
    "min_samples_leaf": [1, 2]         # Min samples in a leaf node
}

print(f"🎯 Hyperparameters to test: {param_grid}")
print(f"   Total combinations: 2 × 3 × 2 × 2 = 24 models\n")

## Step 6d: Connect to MLflow and start tracking
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("House Price Prediction")

## Step 6e: Start an MLflow run (records everything that happens here)
with mlflow.start_run():
    print("🏃 Running GridSearchCV (testing all 24 combinations)...")
    
    # Perform hyperparameter tuning
    grid_search = hyperparameter_tuning(X_train, y_train, param_grid)
    
    # Extract the best model (best performance on validation data)
    best_model = grid_search.best_estimator_
    print(f"\n✅ Best hyperparameters found: {grid_search.best_params_}")
    
    # Step 6f: Evaluate the best model on test data
    print("\n📊 Evaluating best model on test data...")
    y_pred = best_model.predict(X_test)  # Make predictions
    mse = mean_squared_error(y_test, y_pred)  # Calculate error (lower is better)
    rmse = mse ** 0.5
    
    print(f"   Mean Squared Error: {mse:.4f}")
    print(f"   Root Mean Squared Error: {rmse:.4f}")
    
    # Step 6g: Log the best hyperparameters to MLflow
    print("\n📝 Logging hyperparameters to MLflow...")
    mlflow.log_param("n_estimators", grid_search.best_params_["n_estimators"])
    mlflow.log_param("max_depth", grid_search.best_params_["max_depth"])
    mlflow.log_param("min_samples_split", grid_search.best_params_["min_samples_split"])
    mlflow.log_param("min_samples_leaf", grid_search.best_params_["min_samples_leaf"])
    
    # Step 6h: Log the performance metrics
    print("📝 Logging metrics to MLflow...")
    mlflow.log_metric("mean_squared_error", mse)
    mlflow.log_metric("root_mean_squared_error", rmse)
    
    # Step 6i: Save and register the model
    print("📝 Logging model to MLflow...")
    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
    
    if tracking_url_type_store != "file":
        # Register model with MLflow Model Registry (can promote to production later)
        mlflow.sklearn.log_model(best_model, "model", registered_model_name="House Price Predictor")
    else:
        # Save locally if no remote registry available
        mlflow.sklearn.log_model(best_model, "model", signature=signature)
    
    # Step 6j: Print summary
    print("\n" + "="*60)
    print("🎉 EXPERIMENT COMPLETE!")
    print("="*60)
    print(f"✅ Best Hyperparameters: {grid_search.best_params_}")
    print(f"✅ Model MSE on test set: {mse:.4f}")
    print(f"✅ Model RMSE on test set: {rmse:.4f}")
    print(f"\n📊 View all results at: http://127.0.0.1:5000")

📝 Starting the complete machine learning pipeline...

✅ Data split:
   Training samples: 16512
   Testing samples: 4128



2026/03/19 04:52:45 INFO mlflow.tracking.fluent: Experiment with name 'House Price Prediction' does not exist. Creating a new experiment.


🎯 Hyperparameters to test: {'n_estimators': [100, 200], 'max_depth': [5, 10, None], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]}
   Total combinations: 2 × 3 × 2 × 2 = 24 models

🏃 Running GridSearchCV (testing all 24 combinations)...
🔍 Starting GridSearchCV...
   Will test 24 combinations
Fitting 3 folds for each of 24 candidates, totalling 72 fits
✅ GridSearchCV complete!

✅ Best hyperparameters found: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}

📊 Evaluating best model on test data...
   Mean Squared Error: 0.2556
   Root Mean Squared Error: 0.5055

📝 Logging hyperparameters to MLflow...
📝 Logging metrics to MLflow...
📝 Logging model to MLflow...


2026/03/19 04:56:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'House Price Predictor'.
2026/03/19 04:56:33 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: House Price Predictor, version 1
Created version '1' of model 'House Price Predictor'.
2026/03/19 04:56:33 INFO mlflow.tracking._tracking_service.client: 🏃 View run rogue-squid-474 at: http://127.0.0.1:5000/#/experiments/258511750301664723/runs/dbba4c01fea64af9ba286525eb357768.
2026/03/19 04:56:33 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/258511750301664723.



🎉 EXPERIMENT COMPLETE!
✅ Best Hyperparameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
✅ Model MSE on test set: 0.2556
✅ Model RMSE on test set: 0.5055

📊 View all results at: http://127.0.0.1:5000


## 📊 Understanding MLflow Experiment Tracking

### What is MLflow?
MLflow is like a **lab notebook for machine learning** - it records everything about your experiments so you can:

### What MLflow Records:
- **Parameters**: The hyperparameters we used (n_estimators=100, max_depth=5, etc.)
- **Metrics**: Performance measurements (accuracy, loss, error, etc.)
- **Models**: Saves the actual trained model files
- **Artifacts**: Any other files or data we want to keep

### Why This Matters:
✅ **Reproducibility**: You can exactly recreate any previous experiment  
✅ **Comparison**: See which hyperparameters performed best at a glance  
✅ **Deployment**: Register the best model for production use  
✅ **Documentation**: Automatically documents what you tried and what worked  

### What Happened in This Notebook:
1. We tested 24 different hyperparameter combinations (2×3×2×2)
2. MLflow automatically logged each one
3. We found the BEST combination
4. We can now see all 24 experiments in a nice table format
5. We registered the best model so we can use it elsewhere

### View Your Results:
Go to **http://127.0.0.1:5000** and:
- Click on "House Price Prediction" experiment
- See all the runs
- Compare parameters and metrics in a table
- Click on the best run to see full details
- Download the model for use in other code!